In [26]:
import os
from dotenv import load_dotenv
import chromadb
from openai import OpenAI
from IPython.display import Markdown,display

In [4]:
load_dotenv()

groqAPIKey = os.getenv("GroqAPIKey")

In [5]:
client = OpenAI(api_key=groqAPIKey,base_url="https://api.groq.com/openai/v1")

In [6]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role":"user",
            "content":"Tell me about NovaTech Solutions work from home policy"
        }
    ]
)

print(response.choices[0].message.content)

I’m sorry, but I don’t have information about NovaTech Solutions’ work‑from‑home policy. If you need details on that policy, I recommend checking the company’s official employee handbook, intranet site, or contacting the HR department directly.


## Loading and Chunking the data

In [7]:
with open(r"D:\\Veena\\Certificate Program in AI and ML\\Certificate-Program-in-AIML-IIT-Patna\\MODULE 3 - GENAI & AGENTS\\RAG\\company_hr_policy.txt","r") as f:
    hrDoc = f.read()

with open(r"D:\\Veena\\Certificate Program in AI and ML\\Certificate-Program-in-AIML-IIT-Patna\\MODULE 3 - GENAI & AGENTS\\RAG\\product_knowledge_base.txt","r") as f:
    prodDoc = f.read()    

print(f"Length of HR document: {len(hrDoc)}, around {len(hrDoc.split())}")    
print(f"Length of Product document: {len(prodDoc)}, around {len(prodDoc.split())}")    



Length of HR document: 7472, around 1052
Length of Product document: 9824, around 1410


In [8]:
def chunkCreation(text,srcFile):
    paragraphs = text.strip().split("\n\n") # we are splitting the data when two lines are encountered.

    chunks=[]

    for para in paragraphs:
        para = para.strip()


        if para.startswith("==========="):
            continue
        if len(para) < 50:
            continue

        chunks.append({"text":para, "src":srcFile})
    return chunks

In [9]:
hrChunks = chunkCreation(hrDoc,"HR Document")
productChunks = chunkCreation(prodDoc,"Product Document")

In [10]:
# print(f"HR Document chunks: \n{hrChunks}")
# print(f"\n Product Document chunks: \n {productChunks}")

allChunks = hrChunks+productChunks

print(f"Total chunks: {len(allChunks)}")
print(f"HR Chunks: {len(hrChunks)}")
print(f"HR Chunks: {len(productChunks)}")

print(f"Sample Chunk: Source: {allChunks[5]['src']}")
print(f"Sample Chunk: Text: \n{allChunks[5]['text']}")

Total chunks: 58
HR Chunks: 25
HR Chunks: 33
Sample Chunk: Source: HR Document
Sample Chunk: Text: 
Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.


In [11]:
chromaClient = chromadb.Client()

In [12]:
companyCollection=chromaClient.create_collection(name="NovaTechSolutionDocuments")

docs=[]
metadata=[]
chunkIds=[]

for i,chunk in enumerate(allChunks):
    docs.append(chunk['text'])
    chunkIds.append(f"chunk_{i}")
    metadata.append({"source":chunk['src']})

companyCollection.add(ids=chunkIds,documents=docs,metadatas=metadata)

In [13]:
result = companyCollection.query(
    query_texts=['(WFH)'],
    n_results=3,
    include=['embeddings','documents',"distances"]
)

print(f"Query result from chromaDB for query_texts=['(WFH)']: {result}")

Query result from chromaDB for query_texts=['(WFH)']: {'ids': [['chunk_9', 'chunk_16', 'chunk_6']], 'embeddings': [array([[ 0.04135146, -0.02312663, -0.01006458, ...,  0.04163894,
        -0.08532024, -0.01058518],
       [-0.01802602,  0.00306745, -0.03662764, ..., -0.01507321,
         0.08836094, -0.06399044],
       [ 0.03852436, -0.02465586,  0.00420927, ...,  0.0326165 ,
        -0.03194455,  0.01993223]], shape=(3, 384))], 'documents': [['WFH Expectations:\nEmployees must maintain the same productivity levels as in-office work. All meetings must be attended via video call with camera on. Employees must respond to messages within 30 minutes during core hours. Any planned unavailability during WFH must be communicated in advance.', 'Confirmation:\nUpon successful completion of probation, employees receive a confirmation letter and become eligible for all employee benefits including WFH, company health insurance (family floater up to Rs 5 lakhs), and annual performance bonus.', 'Re

In [39]:
result = companyCollection.query(
    query_texts=['Pricing'],
    n_results=3,
    include=['embeddings','documents',"distances"]
)

print(f"Query result from chromaDB for query_texts=['(WFH)']: {result}")

Query result from chromaDB for query_texts=['(WFH)']: {'ids': [['chunk_32', 'chunk_31', 'chunk_30']], 'embeddings': [array([[-0.07316757,  0.0780286 , -0.01051732, ..., -0.08979611,
        -0.0353028 ,  0.03183666],
       [-0.00852375,  0.00845661, -0.0348037 , ..., -0.02174192,
        -0.05913423, -0.0021997 ],
       [ 0.02671012, -0.03611249, -0.03212686, ..., -0.00823624,
        -0.07056292, -0.00987553]], shape=(3, 384))], 'documents': [['Enterprise Plan â€” Custom pricing (contact sales):\n- Everything in Business, plus:\n- Unlimited storage\n- Dedicated account manager\n- 24/7 phone support with 1-hour response SLA\n- On-premise deployment option\n- Custom integrations and API priority\n- Advanced security (SOC 2, HIPAA compliance available)\n- Training and onboarding for teams', 'Business Plan â€” Rs 699/user/month (billed annually) or Rs 899/user/month (billed monthly):\n- Unlimited users\n- Unlimited projects\n- 100 GB storage per user\n- Full task management (Kanban + Li

### RETRIEVAL PIPELINE

In [14]:
def fetchfromVetorDB(userQry,nResults=3):
    dbresp = companyCollection.query(
        query_texts=[userQry],
        n_results= nResults)
    return dbresp['documents'][0],dbresp['metadatas'][0]
    

In [40]:
print(fetchfromVetorDB('Pricing'))

(['Enterprise Plan â€” Custom pricing (contact sales):\n- Everything in Business, plus:\n- Unlimited storage\n- Dedicated account manager\n- 24/7 phone support with 1-hour response SLA\n- On-premise deployment option\n- Custom integrations and API priority\n- Advanced security (SOC 2, HIPAA compliance available)\n- Training and onboarding for teams', 'Business Plan â€” Rs 699/user/month (billed annually) or Rs 899/user/month (billed monthly):\n- Unlimited users\n- Unlimited projects\n- 100 GB storage per user\n- Full task management (Kanban + List + Gantt)\n- Document Hub with version history\n- Time Tracker\n- Priority email and chat support (response within 4 hours)\n- API access\n- Custom workflows and automation\n- SSO (Single Sign-On) support', 'Starter Plan â€” Rs 299/user/month (billed annually) or Rs 399/user/month (billed monthly):\n- Up to 20 users\n- 5 projects\n- 10 GB storage\n- Basic task management (Kanban only)\n- Email support (response within 48 hours)\n- No API acces

In [15]:
docChunks,metadata=fetchfromVetorDB("What is the WFH policy?")
for i in range(len(docChunks)):
    print(f"----------Chunk - {i}----------------")
    print(f"Chunk: {docChunks[i]}")
    print(f"Chunk: {metadata[i]}")

print(f"\nChunks joined: {"\n\n".join(docChunks)}")

----------Chunk - 0----------------
Chunk: WFH Expectations:
Employees must maintain the same productivity levels as in-office work. All meetings must be attended via video call with camera on. Employees must respond to messages within 30 minutes during core hours. Any planned unavailability during WFH must be communicated in advance.
Chunk: {'source': 'HR Document'}
----------Chunk - 1----------------
Chunk: Confirmation:
Upon successful completion of probation, employees receive a confirmation letter and become eligible for all employee benefits including WFH, company health insurance (family floater up to Rs 5 lakhs), and annual performance bonus.
Chunk: {'source': 'HR Document'}
----------Chunk - 2----------------
Chunk: Eligibility:
All employees who have completed their probation period (6 months) are eligible for Work From Home (WFH) arrangements. Employees in their probation period may request WFH only in exceptional circumstances with manager and HR approval.
Chunk: {'source':

### Send this to LLM, with prompt

In [48]:
def askRAG(userQry,n_results=3,verbose=True):
    chunks,source = fetchfromVetorDB(userQry=userQry)

    if verbose:
        print(f"="*60)
        print(f"User question: {userQry}")
        print(f"No.of chunks retrieved from vector DB:{len(chunks)}")
        for i,(chunk,src) in enumerate(zip(chunks,source)):
            print(f"Chunk: \n{chunk[:75]}")
            print(f"Source of chunk: \n{src}")

        print(f"="*60)  

    context = "/n/n".join(chunks)
    # if verbose:
    #     print(f"Chunks: {context}")

    message = [
        {
            "role":"system",
            "content":'''You are a helpful assistant, that will answer only on the provided context. If the
            context doesn't have enough information to anser the user query, reply with "I do not have enough information to answer your question". Do not mapke up information. If the question is about capitals of countries, the n use your intelligence otherwise,  restrict your answer within the provided context.'''
        },
        {
            "role":"user",
            "content":f"Context :{context}, User query: {userQry}"
        }
    ]    

    # if verbose:
    #     print(f"\n message to LLM: \n{message}")


    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=message
    )

    # if verbose:
    #     print(f"\n response: \n{response}")

    answer=response.choices[0].message.content

    if verbose:
        print(f"Answer for the user query: {answer}")

    return answer    

In [30]:
answerLLM = askRAG("What is the WFH policy?",verbose=False)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

**Work‑From‑Home (WFH) Policy**

**Eligibility**
- All employees who have successfully completed their 6‑month probation period are eligible for WFH arrangements.  
- Employees who are still in their probation period may request WFH only in exceptional circumstances, and such requests must be approved by their manager and HR.

**Expectations for Eligible Employees**
1. **Productivity** – Maintain the same productivity levels as when working in the office.  
2. **Meetings** – Attend all meetings via video call with the camera turned on.  
3. **Responsiveness** – Respond to messages within 30 minutes during core working hours.  
4. **Planned Unavailability** – Any planned periods of unavailability while working from home must be communicated in advance.

In [34]:
answerLLM = askRAG("How many days of annual leaves does employees get?",verbose=True)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

User question: How many days of annual leaves does employees get?
No.of chunks retrieved from vector DB:3
Chunk: 
Annual Leave:
All full-time employees are entitled to 24 days of paid annua
Source of chunk: 
{'source': 'HR Document'}
Chunk: 
Casual Leave:
Employees are entitled to 6 days of casual leave per year. Ca
Source of chunk: 
{'source': 'HR Document'}
Chunk: 
Sick Leave:
Employees are entitled to 12 days of sick leave per year. Sick 
Source of chunk: 
{'source': 'HR Document'}
Answer for the user query: Employees are entitled to **24 days of paid annual leave per calendar year**.


Employees are entitled to **24 days of paid annual leave per calendar year**.

In [36]:
answerLLM = askRAG("What happens during the probation period?",verbose=False)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

During the probation period (the first 6 months from the date of joining):

- **Termination rights:** Either the employee or the employer can end the employment by giving 15 days’ notice, or by paying salary in lieu of that notice.
- **Performance review & possible extension:** If an employee’s performance is considered borderline, the probation may be extended for up to an additional 3 months. This extension requires approval from HR and the department head and must be communicated to the employee in writing.
- **Work‑from‑home (WFH) eligibility:** Employees are not eligible for regular WFH arrangements while on probation. They may request WFH only under exceptional circumstances, and such requests need approval from both their manager and HR.

In [45]:
answerLLM = askRAG("what are pricing details of CloudDesk Pro?",verbose=True)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

User question: what are pricing details of CloudDesk Pro?
No.of chunks retrieved from vector DB:3
Chunk: 
CloudDesk Pro â€” Product Knowledge Base
Internal Support Reference | Versi
Source of chunk: 
{'source': 'Product Document'}
Chunk: 
Support Channels:
- Help Center: docs.clouddesk.pro â€” Self-service articl
Source of chunk: 
{'source': 'Product Document'}
Chunk: 
What is CloudDesk Pro?
CloudDesk Pro is a cloud-based project management an
Source of chunk: 
{'source': 'Product Document'}
Answer for the user query: I do not have enough information to answer your question.


I do not have enough information to answer your question.

In [46]:
answerLLM = askRAG("what are pricing details?",verbose=True)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

User question: what are pricing details?
No.of chunks retrieved from vector DB:3
Chunk: 
Enterprise Plan â€” Custom pricing (contact sales):
- Everything in Busines
Source of chunk: 
{'source': 'Product Document'}
Chunk: 
Business Plan â€” Rs 699/user/month (billed annually) or Rs 899/user/month 
Source of chunk: 
{'source': 'Product Document'}
Chunk: 
Starter Plan â€” Rs 299/user/month (billed annually) or Rs 399/user/month (
Source of chunk: 
{'source': 'Product Document'}
Answer for the user query: **Pricing details from the provided context**

| Plan | Billing option | Price per user |
|------|----------------|----------------|
| **Starter Plan** | Annually | Rs 299 / user / month |
|  | Monthly | Rs 399 / user / month |
| **Business Plan** | Annually | Rs 699 / user / month |
|  | Monthly | Rs 899 / user / month |
| **Enterprise Plan** | Custom pricing (contact sales) | Not listed – you need to contact the sales team for a quote. |


**Pricing details from the provided context**

| Plan | Billing option | Price per user |
|------|----------------|----------------|
| **Starter Plan** | Annually | Rs 299 / user / month |
|  | Monthly | Rs 399 / user / month |
| **Business Plan** | Annually | Rs 699 / user / month |
|  | Monthly | Rs 899 / user / month |
| **Enterprise Plan** | Custom pricing (contact sales) | Not listed – you need to contact the sales team for a quote. |

### From the above, we have to observe one thing, both the questions are similar, related to "pricing details". In one question, I have included "CloudDesk Pro". Now my chunks, does not have "cloudDesk Pro" and pricing details combined. So when I asked "Pricing details of CloudDesk Pro", the model might not be able to retrieve the relevant information because it doesn't have the context of "CloudDesk Pro" in the chunks. But w/o "CloudDesk Pro" in the question, it retrieved information about pricing details. this is a limitation of Naive-RAG. This can be solved by using overlap chunking, including meta data etc.

In [49]:
answerLLM = askRAG("what are pricing detailsHow to cancel my subscription?",verbose=False)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

**How to cancel your subscription**  
1. Open **Settings** → **Billing** → **Subscription** → **Cancel**.  
2. You’ll be asked to confirm the cancellation and provide a reason.  
3. Your account stays active until the end of the current billing period, and you can reactivate any time before that period ends.

**Pricing‑related details from the provided information**

- **Annual Billing Discount**: All plans give roughly a **25 % discount** when you choose annual billing instead of monthly billing.  
- **Refund Policy**  
  - *Monthly subscriptions*: Full refund if you cancel within **48 hours** of the charge; no refund after that.  
  - *Annual subscriptions*: Generally **non‑refundable**, but if you cancel within the first **30 days** of a new annual term, you may request a prorated refund (the amount is calculated at the monthly rate, minus one month).

No specific price amounts are included in the context, so the above are the only pricing‑related details available.

In [52]:
answerLLM = askRAG("what is the capital of France?",verbose=False)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

Paris.

In [53]:
answerLLM = askRAG("what is the recepie of Tea?",verbose=False)
#print(f"Answer for LLM: \n {answerLLM}")
display(Markdown(answerLLM))

I do not have enough information to answer your question.

### What are challenges in RAG?
- Poor chunking strategy
- The no.of chunks to retrieve is always dynamic, it depends on the question. If the question is very specific, then we need to retrieve less chunks. If the question is generic, then we need to retrieve more chunks. So we need to have a dynamic retrieval strategy.

### CAG - Cached Augmented Generation
- Here, the frequntly asked questions and their answers are cached. So when a question is asked, it first checks in the cache, if it is present, then it returns the answer from the cache. If not, then it goes to the RAG pipeline to get the answer. This is useful for frequently asked questions.